In [0]:
SELECT *,
  CASE
    WHEN Date RLIKE '^[A-Za-z]{3}-[0-9]{1,2}-[0-9]{4}$' THEN to_date(Date, 'MMM-d-yyyy')
    WHEN Date RLIKE '^[0-9]{1,2}/[0-9]{1,2}/[0-9]{2}$' THEN to_date(Date, 'M/d/yy')
    ELSE NULL
  END AS Date_fixed,
  CASE
    WHEN Smoker = 'yes' THEN True
    WHEN Smoker IN ('no', 'n') THEN False
    ELSE NULL
  END AS Smoker_fixed,
  CASE
    WHEN `Heart Rate` = 1600 THEN (SELECT MEDIAN(`Heart Rate`) FROM workspace.default.health_irreg_rhythm WHERE `Heart Rate` != 1600)
    ELSE `Heart Rate`
  END AS Heart_Rate_fixed,
  CASE
    WHEN `Rings Closed` = 4 THEN 3
    ELSE `Rings Closed`
  END AS Rings_Closed_fixed
FROM workspace.default.health_irreg_rhythm

In [0]:
SELECT Id, Cohort, COUNT(Id)
FROM health_irreg_rhythm
GROUP BY Cohort, Id
HAVING COUNT(Id) != 3
Order BY Cohort, Id;


In [0]:
SELECT Cohort, COUNT(Id)
FROM health_irreg_rhythm
GROUP BY Cohort;

In [0]:
CREATE OR REPLACE VIEW health_irreg_rhythm_fixed AS
(
SELECT *,
  CASE
    WHEN Date RLIKE '^[A-Za-z]{3}-[0-9]{1,2}-[0-9]{4}$' THEN to_date(Date, 'MMM-d-yyyy')
    WHEN Date RLIKE '^[0-9]{1,2}/[0-9]{1,2}/[0-9]{2}$' THEN to_date(Date, 'M/d/yy')
    ELSE NULL
  END AS Date_fixed,
  CASE
    WHEN Smoker = 'yes' THEN True
    WHEN Smoker IN ('no', 'n') THEN False
    ELSE NULL
  END AS Smoker_fixed,
  CASE
    WHEN `Heart Rate` = 1600 THEN (SELECT MEDIAN(`Heart Rate`) FROM workspace.default.health_irreg_rhythm WHERE `Heart Rate` != 1600)
    ELSE `Heart Rate`
  END AS Heart_Rate_fixed,
  CASE
    WHEN `Rings Closed` = 4 THEN 3
    ELSE `Rings Closed`
  END AS Rings_Closed_fixed,
  CASE
    WHEN Cohort = 3 AND Id = 3 THEN 4
    ELSE Cohort
  END AS Cohort_fixed
FROM workspace.default.health_irreg_rhythm
)

In [0]:
WITH smoking_timeline AS (
    SELECT 
        Id,
        Date_fixed,
        Smoker_fixed,
        ROW_NUMBER() OVER (PARTITION BY Id ORDER BY Date_fixed) as timepoint,
        COUNT(*) OVER (PARTITION BY Id) as total_timepoints
    FROM health_irreg_rhythm_fixed
),
smoking_summary AS (
    SELECT 
        Id,
        total_timepoints,
        MAX(CASE WHEN timepoint = 1 THEN smoker_fixed END) as first_status,
        MAX(CASE WHEN timepoint = 2 THEN smoker_fixed END) as middle_status,
        MAX(CASE WHEN timepoint = total_timepoints THEN smoker_fixed END) as last_status,
        SUM(CASE WHEN smoker_fixed = 'yes' THEN 1 ELSE 0 END) as times_smoked
    FROM smoking_timeline
    GROUP BY id, total_timepoints
),
categorized AS (
    SELECT 
        CASE 
            -- 1. Did not change
            WHEN times_smoked = 0 OR times_smoked = total_timepoints 
                THEN 'Did Not Change'
            -- 2. Started smoking
            WHEN first_status = 'no' AND last_status = 'yes'
                THEN 'Started Smoking'
            -- 3. Quit smoking
            WHEN first_status = 'yes' AND last_status = 'no'
                THEN 'Quit Smoking'
            -- 4. Started then quit
            WHEN first_status = 'no' AND middle_status = 'yes' AND last_status = 'no'
                THEN 'Started Then Quit'
            -- 5. Quit then restarted
            WHEN first_status = 'yes' AND middle_status = 'no' AND last_status = 'yes'
                THEN 'Quit Then Restarted'
            ELSE 'Other'
        END as smoking_category
    FROM smoking_summary
)
SELECT 
    smoking_category,
    COUNT(*) as participant_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(DISTINCT id) FROM health_irreg_rhythm), 1) 
        as percent_of_participants
FROM categorized
GROUP BY smoking_category;